# GHIA — Ferramental Quantitativo | Fase 0: Infraestrutura de Coleta de Dados

Protótipo de coleta e tratamento das séries macro e de mercado que servem de base para as fases
seguintes do projeto (faixa de erro histórico do Focus, distribuição condicional de retorno por
classe de ativo, Black-Litterman).

**Fontes:** SGS/Banco Central, SIDRA/IBGE, Boletim Focus (BCB) e IPEADATA.

**Saída:** base mensal única em Parquet, com log de execução e checagem de consistência.

In [1]:
from bcb import sgs
import sidrapy
import ipeadatapy as ipea
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import json
import logging
from pathlib import Path

pd.set_option('display.width', 120)

## 0. Configuração geral (pastas, log, parâmetros)

In [2]:
# Pastas de saída. DATA_DIR guarda a base tratada; RAW_DIR guarda o dado bruto de cada fonte
# (útil para auditar uma coleta que deu errado sem precisar buscar tudo de novo).

DATA_DIR = Path("dados_ghia")
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

LOG_PATH = DATA_DIR / "log_atualizacao.jsonl"
DATA_INICIO = "2003-01-01"  # início da série histórica

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ghia_coleta")


def registrar_log(etapa: str, status: str, detalhes: dict | None = None) -> None:
    """
    Registra uma linha de log estruturado (JSON Lines) com o resultado de uma etapa de coleta.

    `status` deve ser "ok" ou "erro". O arquivo acumula um histórico de execuções, permitindo
    checar depois quando cada fonte foi atualizada com sucesso pela última vez.
    """
    registro = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "etapa": etapa,
        "status": status,
        "detalhes": detalhes or {},
    }
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    nivel = logging.INFO if status == "ok" else logging.WARNING
    logger.log(nivel, f"[{etapa}] {status} | {detalhes or ''}")

## 1. Coleta — SGS (Banco Central)

Séries diárias e mensais do Sistema Gerenciador de Séries Temporais. Códigos conferidos
diretamente na API (`api.bcb.gov.br`) em 2026-08-29.

In [3]:
# (código, é_diária) — o SGS limita consultas de série diária a janelas de no máximo
# 10 anos; séries mensais não têm esse limite.
CODIGOS_SGS = {
    "selic_meta": (432, True),              # Meta Selic definida pelo Copom (% a.a.), diária
    "selic_over_mensal": (4390, False),     # Selic acumulada no mês, anualizada (% a.a.)
    "ipca_mensal": (433, False),            # IPCA - variação mensal (%)
    "ipca_12m": (13522, False),             # IPCA - variação acumulada em 12 meses (%)
    "cambio_compra": (1, True),             # Dólar americano (compra), câmbio livre, diário
    "ibcbr_dessaz": (24364, False),         # IBC-Br, com ajuste sazonal
    "credito_saldo_total": (20539, False),  # Saldo da carteira de crédito - total (R$ milhões)
}


def _dividir_em_janelas(inicio: pd.Timestamp, fim: pd.Timestamp, anos: int = 9) -> list[tuple]:
    """Quebra um intervalo de datas em janelas de até `anos` anos (fecho à direita inclusive)."""
    janelas = []
    cursor = inicio
    while cursor < fim:
        fim_janela = min(cursor + pd.DateOffset(years=anos, days=-1), fim)
        janelas.append((cursor, fim_janela))
        cursor = fim_janela + pd.DateOffset(days=1)
    return janelas


def _coletar_serie_sgs(nome: str, codigo: int, data_inicio: str, diaria: bool) -> pd.DataFrame:
    """
    Coleta uma única série do SGS. Uma série por chamada (em vez de várias num só
    `sgs.get`) — testado e mais estável que pedir várias séries de uma vez, que se
    mostrou instável nesta rede. Séries diárias são buscadas em janelas de até 9 anos
    para respeitar o limite de 10 anos da API.
    """
    inicio, fim = pd.Timestamp(data_inicio), pd.Timestamp.today()
    janelas = _dividir_em_janelas(inicio, fim) if diaria else [(inicio, fim)]
    partes = [sgs.get({nome: codigo}, start=str(i.date()), end=str(f.date())) for i, f in janelas]
    serie = pd.concat(partes).sort_index()
    return serie[~serie.index.duplicated(keep="last")]


def coletar_sgs(codigos: dict, data_inicio: str = DATA_INICIO) -> pd.DataFrame:
    """
    Coleta múltiplas séries do SGS/Banco Central e retorna um único DataFrame indexado
    por data (uma coluna por série). Séries de frequências diferentes (diária, mensal)
    ficam com NaN nas datas em que não há observação — o alinhamento de calendário é
    responsabilidade da etapa de tratamento, não da coleta.
    """
    colunas = [_coletar_serie_sgs(nome, codigo, data_inicio, diaria) for nome, (codigo, diaria) in codigos.items()]
    df = pd.concat(colunas, axis=1)
    df.index.name = "data"
    return df


try:
    df_sgs = coletar_sgs(CODIGOS_SGS)
    df_sgs.to_parquet(RAW_DIR / "sgs.parquet")
    registrar_log("coleta_sgs", "ok", {"n_series": len(CODIGOS_SGS), "n_obs": len(df_sgs)})
except Exception as e:
    df_sgs = pd.DataFrame()
    registrar_log("coleta_sgs", "erro", {"mensagem": str(e)})

df_sgs.tail()

2026-08-29 21:15:24,299 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=31%2F12%2F2011 "HTTP/1.1 200 OK"


2026-08-29 21:15:25,337 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=01%2F01%2F2012&dataFinal=31%2F12%2F2020 "HTTP/1.1 200 OK"


2026-08-29 21:15:25,909 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial=01%2F01%2F2021&dataFinal=29%2F08%2F2026 "HTTP/1.1 200 OK"


2026-08-29 21:15:26,186 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.4390/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=29%2F08%2F2026 "HTTP/1.1 200 OK"


2026-08-29 21:15:26,435 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=29%2F08%2F2026 "HTTP/1.1 200 OK"


...

2026-08-29 21:15:28,286 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados?formato=json&dataInicial=01%2F01%2F2021&dataFinal=29%2F08%2F2026 "HTTP/1.1 200 OK"


2026-08-29 21:15:28,548 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.24364/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=29%2F08%2F2026 "HTTP/1.1 200 OK"


2026-08-29 21:15:28,805 | INFO | HTTP Request: GET https://api.bcb.gov.br/dados/serie/bcdata.sgs.20539/dados?formato=json&dataInicial=01%2F01%2F2003&dataFinal=29%2F08%2F2026 "HTTP/1.1 200 OK"


2026-08-29 21:15:28,889 | INFO | [coleta_sgs] ok | {'n_series': 7, 'n_obs': 8642}


,selic_meta,selic_over_mensal,ipca_mensal,ipca_12m,cambio_compra,ibcbr_dessaz,credito_saldo_total
data,,,,,,,
2026-08-25,14.0,NaN,NaN,NaN,5.1490,NaN,NaN
2026-08-26,14.0,NaN,NaN,NaN,5.1604,NaN,NaN
2026-08-27,14.0,NaN,NaN,NaN,5.1642,NaN,NaN
2026-08-28,14.0,NaN,NaN,NaN,5.2005,NaN,NaN
2026-08-29,14.0,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Coleta — SIDRA (IBGE)

Taxa de desocupação da PNAD Contínua (trimestre móvel), tabela 6381, agregado Brasil.

In [4]:
def coletar_sidra_desocupacao() -> pd.DataFrame:
    """
    Coleta a série completa da taxa de desocupação (PNAD Contínua, trimestre móvel,
    Brasil) via SIDRA e retorna um DataFrame mensal com a taxa em pontos percentuais.
    A data atribuída a cada trimestre móvel é o último mês do trimestre.
    """
    bruto = sidrapy.get_table(
        table_code="6381",
        territorial_level="1",
        ibge_territorial_code="all",
        variable="4099",
        period="all",
    )
    bruto = bruto.iloc[1:].copy()  # primeira linha é o cabeçalho descritivo (D2C = Trimestre Móvel)
    bruto["data"] = pd.to_datetime(bruto["D2C"], format="%Y%m")
    bruto["taxa_desocupacao"] = pd.to_numeric(bruto["V"], errors="coerce")
    return bruto.set_index("data")[["taxa_desocupacao"]].sort_index()


try:
    df_sidra = coletar_sidra_desocupacao()
    df_sidra.to_parquet(RAW_DIR / "sidra_desocupacao.parquet")
    registrar_log("coleta_sidra", "ok", {"n_obs": len(df_sidra)})
except Exception as e:
    df_sidra = pd.DataFrame()
    registrar_log("coleta_sidra", "erro", {"mensagem": str(e)})

df_sidra.tail()

2026-08-29 21:15:30,350 | INFO | [coleta_sidra] ok | {'n_obs': 173}


,taxa_desocupacao
data,
2026-03-01,6.1
2026-04-01,5.8
2026-05-01,5.6
2026-06-01,5.4
2026-07-01,5.3


## 3. Coleta — Boletim Focus (expectativas de mercado, BCB)

Serviço OData `Expectativas` do BCB (Olinda). Duas séries relevantes para a Fase 1
(faixa de erro histórico do Focus):

- `ExpectativaMercadoMensais`: mediana/média por indicador e mês de referência.
- `ExpectativasMercadoInflacao12Meses`: mediana/média da inflação acumulada nos
  próximos 12 meses — é a série de horizonte fixo que permite medir erro por horizonte
  sem ter que reconstruir o "horizonte" a partir da data de referência.

In [5]:
FOCUS_BASE_URL = "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata"


def _consultar_focus_odata(entidade: str, filtro: str, orderby: str = "Data") -> pd.DataFrame:
    """
    Faz uma consulta genérica ao serviço OData de Expectativas do BCB, paginando em
    blocos de 1000 registros (limite do serviço) até esgotar o resultado.
    """
    registros = []
    skip = 0
    tamanho_pagina = 1000
    while True:
        url = (
            f"{FOCUS_BASE_URL}/{entidade}"
            f"?$filter={filtro}&$orderby={orderby}&$format=json"
            f"&$top={tamanho_pagina}&$skip={skip}"
        )
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        pagina = resp.json()["value"]
        registros.extend(pagina)
        if len(pagina) < tamanho_pagina:
            break
        skip += tamanho_pagina
    return pd.DataFrame(registros)


def coletar_focus_ipca_12m(suavizada: str = "N") -> pd.DataFrame:
    """
    Coleta a série de expectativa de IPCA acumulado em 12 meses à frente (horizonte fixo),
    versão não suavizada por padrão. Retorna média, mediana e desvio-padrão por data de
    coleta, indexado por data.
    """
    filtro = f"Indicador eq 'IPCA' and Suavizada eq '{suavizada}'"
    df = _consultar_focus_odata("ExpectativasMercadoInflacao12Meses", filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    return df.set_index("Data").sort_index()


def coletar_focus_mensal(indicador: str = "IPCA", anos_recentes: int = 2) -> pd.DataFrame:
    """
    Coleta a série de expectativas mensais (por mês de referência) para um indicador do
    Focus (ex.: IPCA, Câmbio, Selic). Cada linha é uma combinação (data de coleta, mês de
    referência) — granularidade fina, útil para horizontes diferentes de 12 meses.

    Limitada aos últimos `anos_recentes` anos por desempenho: essa tabela tem frequência
    diária cruzada com dezenas de meses de referência, e o serviço do BCB pagina devagar
    em consultas longas (~1.5s por 1000 linhas). Para o histórico completo em produção,
    rodar esta função como job em lote separado, fora deste notebook interativo.
    """
    data_minima = (pd.Timestamp.today() - pd.DateOffset(years=anos_recentes)).strftime("%Y-%m-%d")
    filtro = f"Indicador eq '{indicador}' and Data ge '{data_minima}'"
    df = _consultar_focus_odata("ExpectativaMercadoMensais", filtro)
    df["Data"] = pd.to_datetime(df["Data"])
    df["DataReferencia"] = pd.to_datetime(df["DataReferencia"], format="%m/%Y")
    return df.sort_values(["DataReferencia", "Data"])


try:
    df_focus_12m = coletar_focus_ipca_12m()
    df_focus_12m.to_parquet(RAW_DIR / "focus_ipca_12m.parquet")
    registrar_log("coleta_focus_12m", "ok", {"n_obs": len(df_focus_12m)})
except Exception as e:
    df_focus_12m = pd.DataFrame()
    registrar_log("coleta_focus_12m", "erro", {"mensagem": str(e)})

try:
    df_focus_ipca_mensal = coletar_focus_mensal("IPCA")
    df_focus_ipca_mensal.to_parquet(RAW_DIR / "focus_ipca_mensal.parquet")
    registrar_log("coleta_focus_mensal", "ok", {"n_obs": len(df_focus_ipca_mensal)})
except Exception as e:
    df_focus_ipca_mensal = pd.DataFrame()
    registrar_log("coleta_focus_mensal", "erro", {"mensagem": str(e)})

df_focus_12m.tail()

2026-08-29 21:15:45,157 | INFO | [coleta_focus_12m] ok | {'n_obs': 9395}


2026-08-29 21:16:25,319 | INFO | [coleta_focus_mensal] ok | {'n_obs': 24900}


,Indicador,Suavizada,Media,Mediana,DesvioPadrao,Minimo,Maximo,numeroRespondentes,baseCalculo
Data,,,,,,,,,
2026-08-19,IPCA,N,4.3239,4.3714,0.3656,3.2138,5.0622,56.0,1
2026-08-20,IPCA,N,4.2633,4.3043,0.4866,2.6940,5.5933,134.0,0
2026-08-20,IPCA,N,4.3277,4.3711,0.3616,3.2138,5.0622,52.0,1
2026-08-21,IPCA,N,4.2789,4.3086,0.4989,2.6940,5.9637,135.0,0
2026-08-21,IPCA,N,4.3554,4.3734,0.4914,2.6940,5.9637,91.0,1


## 4. Coleta — IPEADATA

Fonte adicional mencionada no briefing. A API pública do IPEADATA é historicamente
instável (fora do ar com frequência) — por isso a coleta é isolada em try/except e não
derruba o pipeline caso falhe; a base tratada final simplesmente sai sem essas colunas
naquela execução, e o log registra o problema para investigar depois.

In [6]:
CODIGOS_IPEADATA = {
    "ipca_ipea_12m": "PRECOS12_IPCAGA12",      # IPCA, var. acumulada 12 meses (%) — cruzamento com SGS
    "expectativa_ipca_ipea": "BM12_IPCAEXP1212",  # Expectativa média de inflação 12 meses (%)
}


def coletar_ipeadata(codigos: dict) -> pd.DataFrame:
    """
    Coleta múltiplas séries do IPEADATA e as consolida em um único DataFrame mensal.
    Mantém apenas a última coluna de cada série retornada pela biblioteca (valor numérico),
    descartando metadados redundantes.
    """
    colunas = []
    for nome, codigo in codigos.items():
        serie = ipea.timeseries(codigo).iloc[:, [-1]]
        serie.columns = [nome]
        colunas.append(serie)
    return pd.concat(colunas, axis=1)


try:
    df_ipea = coletar_ipeadata(CODIGOS_IPEADATA)
    df_ipea.to_parquet(RAW_DIR / "ipeadata.parquet")
    registrar_log("coleta_ipeadata", "ok", {"n_series": len(CODIGOS_IPEADATA), "n_obs": len(df_ipea)})
except Exception as e:
    df_ipea = pd.DataFrame()
    registrar_log("coleta_ipeadata", "erro", {"mensagem": str(e)})

df_ipea.tail()

2026-08-29 21:16:40,390 | WARNING | [coleta_ipeadata] erro | {'mensagem': "'NoneType' object has no attribute 'rename'"}


""


## 5. Tratamento — consolidação em base mensal única

Regra de alinhamento: séries diárias (Selic meta, câmbio) viram média mensal;
séries que já nascem mensais são apenas reindexadas para o primeiro dia do mês.
Nenhuma dessazonalização é aplicada aqui — o IBC-Br e a taxa de desocupação já vêm
dessazonalizados na fonte; séries que precisarem de ajuste específico por classe de
ativo entram nessa etapa nas fases seguintes, não na infraestrutura de coleta.

In [7]:
SERIES_DIARIAS = [nome for nome, (_, diaria) in CODIGOS_SGS.items() if diaria]


def mensualizar(df: pd.DataFrame, colunas_diarias: list[str]) -> pd.DataFrame:
    """
    Recebe um DataFrame com índice de data (frequência mista) e devolve uma versão
    mensal: colunas em `colunas_diarias` são agregadas pela média do mês; as demais
    colunas são reamostradas por último valor não nulo do mês (já são mensais na origem).
    """
    diarias = df[colunas_diarias].resample("MS").mean()
    mensais = df.drop(columns=colunas_diarias).resample("MS").last()
    return diarias.join(mensais, how="outer")


df_sgs_mensal = mensualizar(df_sgs, SERIES_DIARIAS) if not df_sgs.empty else pd.DataFrame()
df_sidra_mensal = df_sidra.resample("MS").last() if not df_sidra.empty else pd.DataFrame()

base = df_sgs_mensal.join(df_sidra_mensal, how="outer")
if not df_ipea.empty:
    df_ipea.index = pd.to_datetime(df_ipea.index)
    base = base.join(df_ipea.resample("MS").last(), how="outer")

base = base.sort_index()
base.tail()

,selic_meta,cambio_compra,selic_over_mensal,ipca_mensal,ipca_12m,ibcbr_dessaz,credito_saldo_total,taxa_desocupacao
data,,,,,,,,
2026-04-01,14.741667,5.033075,1.09,0.67,4.39,110.90086,7259766.0,5.8
2026-05-01,14.500000,4.983700,1.07,0.58,4.72,110.93102,7305306.0,5.6
2026-06-01,14.391667,5.127571,1.12,0.16,4.64,110.22154,7353293.0,5.4
2026-07-01,14.250000,5.113948,1.22,0.07,4.44,NaN,7372243.0,5.3
2026-08-01,14.043103,5.151740,1.04,NaN,NaN,NaN,NaN,NaN


## 6. Checagem de consistência

In [8]:
def checar_consistencia(df: pd.DataFrame) -> dict:
    """
    Roda checagens básicas de qualidade sobre a base consolidada: percentual de nulos
    por coluna, existência de datas duplicadas e maior gap (em meses) sem nenhuma
    observação registrada em pelo menos uma série. Não corrige nada — só relata.
    """
    duplicadas = int(df.index.duplicated().sum())
    nulos_pct = (df.isna().mean() * 100).round(1).to_dict()

    calendario_completo = pd.date_range(df.index.min(), df.index.max(), freq="MS")
    meses_faltantes = calendario_completo.difference(df.index)

    return {
        "periodo": [str(df.index.min().date()), str(df.index.max().date())],
        "n_linhas": len(df),
        "datas_duplicadas": duplicadas,
        "nulos_pct_por_coluna": nulos_pct,
        "meses_faltantes_no_indice": [str(d.date()) for d in meses_faltantes],
    }


relatorio = checar_consistencia(base)
registrar_log("checagem_consistencia", "ok", relatorio)
relatorio

2026-08-29 21:16:40,429 | INFO | [checagem_consistencia] ok | {'periodo': ['2003-01-01', '2026-08-01'], 'n_linhas': 284, 'datas_duplicadas': 0, 'nulos_pct_por_coluna': {'selic_meta': 0.0, 'cambio_compra': 0.0, 'selic_over_mensal': 0.0, 'ipca_mensal': 0.4, 'ipca_12m': 0.4, 'ibcbr_dessaz': 0.7, 'credito_saldo_total': 0.4, 'taxa_desocupacao': 39.1}, 'meses_faltantes_no_indice': []}


{'periodo': ['2003-01-01', '2026-08-01'],
 'n_linhas': 284,
 'datas_duplicadas': 0,
 'nulos_pct_por_coluna': {'selic_meta': 0.0,
  'cambio_compra': 0.0,
  'selic_over_mensal': 0.0,
  'ipca_mensal': 0.4,
  'ipca_12m': 0.4,
  'ibcbr_dessaz': 0.7,
  'credito_saldo_total': 0.4,
  'taxa_desocupacao': 39.1},
 'meses_faltantes_no_indice': []}

## 7. Exportação da base tratada

In [9]:
CAMINHO_BASE = DATA_DIR / "base_ghia_mensal.parquet"

base.to_parquet(CAMINHO_BASE)
registrar_log("exportacao_base", "ok", {"caminho": str(CAMINHO_BASE), "n_linhas": len(base), "n_colunas": base.shape[1]})

print(f"Base exportada: {CAMINHO_BASE} ({base.shape[0]} linhas x {base.shape[1]} colunas)")
base.tail()

2026-08-29 21:16:40,440 | INFO | [exportacao_base] ok | {'caminho': 'dados_ghia/base_ghia_mensal.parquet', 'n_linhas': 284, 'n_colunas': 8}


Base exportada: dados_ghia/base_ghia_mensal.parquet (284 linhas x 8 colunas)


,selic_meta,cambio_compra,selic_over_mensal,ipca_mensal,ipca_12m,ibcbr_dessaz,credito_saldo_total,taxa_desocupacao
data,,,,,,,,
2026-04-01,14.741667,5.033075,1.09,0.67,4.39,110.90086,7259766.0,5.8
2026-05-01,14.500000,4.983700,1.07,0.58,4.72,110.93102,7305306.0,5.6
2026-06-01,14.391667,5.127571,1.12,0.16,4.64,110.22154,7353293.0,5.4
2026-07-01,14.250000,5.113948,1.22,0.07,4.44,NaN,7372243.0,5.3
2026-08-01,14.043103,5.151740,1.04,NaN,NaN,NaN,NaN,NaN
